In [ ]:
import os

import numpy as np
import pandas as pd
import xarray as xr
import scipy.io as sio
import matplotlib.pyplot as plt

# settings
%config InlineBackend.figure_format = 'retina'

# Proxy data. Override with PROXY_DATA_DIR; see config/paths.env.example.
# The default is the repo-relative tree, so this notebook runs with no setup provided it is launched from the repo root.
dpath0 = os.environ.get('PROXY_DATA_DIR', 'data/raw')
# Bacon sample-depth registers still live under the proxy tree, unlike the published records.
dpath1 = f'{dpath0}/DSDP-480-479/age_model'
# Repo-tracked external data (LR04 benthic stack, d18O, and pollen records).
extern = 'data/external'
# save figs here
opath = os.environ.get('FIG_OUTPUT_DIR', 'outputs')
os.makedirs(opath, exist_ok=True)

**DSDP 480/479 BACON age-depth models**

In [ ]:
# DSDP-480 depths
dsdp480_depths=pd.read_excel(f'{dpath1}/sample_depths_480.xlsx').depth.values # cm

# DSDP-480 bacon inputs
dsdp480_input=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP480/DSDP480.csv')
xerr480 = dsdp480_input.error.values  # x-axis error

# DSDP-480 bacon ensemble.
# CANONICAL age model for DSDP-480 is Bacon_runs/DSDP480 — see DATA_MANIFEST.md section 2.
# This ensemble is the one behind DSDP480_165_ages.txt, which the MATLAB pipeline
# (dDwax_data_processing_d480_d479.m) uses to assign sample ages: its column medians match
# that file's `median` to 0.5 yr, and its 2.5/97.5 column percentiles match `min`/`max` to 0.5 yr.
dsdp480_mcmc=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP480/DSDP480_mcmc_new.csv', header=None)
ens_num=np.linspace(1,len(dsdp480_mcmc),len(dsdp480_mcmc))
dims = ['ensemble_number','depth']
coords = {'ensemble_number': ens_num,
          'depth': dsdp480_depths}
dsdp480_agedepth=xr.DataArray(dsdp480_mcmc.values, dims=dims, coords=coords) # each row is an age model
median480=dsdp480_agedepth.median(axis=0)

# Confidence bands: percentiles of the ensemble at each depth, ordered [2.5, 16, 84, 97.5].
# Percentiles rather than indices into a sorted array so this stays correct at any ensemble width
p480 = np.nanpercentile(dsdp480_mcmc.values, [2.5, 16, 84, 97.5], axis=0)


# ---------------------------------------------------------------------------------------------
# PUT THE TIE POINTS ON THE SAME TIMESCALE AS THE AGE MODEL BEFORE PLOTTING THEM.
#
# DSDP480.csv stores each date in whatever units Bacon expects as INPUT, and those units are not
# the same for every row. The `cc` column selects the calibration curve:
#
#   cc = 0  ->  `age` is ALREADY a calendar age. Bacon takes it as given. (rows 0-5, 12-16:
#               the Murray/Barron magnetic-susceptibility ties, the SH82 and d479 tie points)
#   cc = 2  ->  `age` is an UNCALIBRATED radiocarbon age. Bacon calibrates it internally against
#               the marine curve, shifted by the local reservoir offset `dR` (300 +/- 20 yr).
#               (rows 6-11: the six Keigwin & Jones planktic dates)
#
# The age-depth model Bacon returns is in CALENDAR years BP. So plotting a cc=2 row's raw `age`
# against that model puts a radiocarbon year on a calendar-year axis. 

# The cc=2 markers plotting locations are MODEL-DEPENDENT. They sit on the median line by construction
# and are NOT an independent check of the fit; they only show which depths carry radiocarbon control.
# The cc=0 markers are still drawn at their own input ages and remain genuine independent checks of the model.
d480_udepth, _first = np.unique(dsdp480_depths, return_index=True)  # 2390 cm is duplicated

def _on_agemodel(per_depth_values, depths=None):
    """Interpolate a per-depth age-model quantity onto depths (cm).

    Defaults to the Bacon tie-point depths; pass `depths` for anything else.
    """
    if depths is None:
        depths = dsdp480_input.depth.values
    return np.interp(depths, d480_udepth, per_depth_values[_first])

is_c14 = (dsdp480_input.cc.values == 2)          # the rows Bacon calibrated internally
# Calendar-age position of every tie point:
# Bacon's posterior median for cc=2, the input age for cc=0 (which is already calendar)
d480_tie_age = np.where(is_c14, _on_agemodel(median480.values), dsdp480_input.age.values)
# Error bars: Bacon's 95% interval for cc=2, the reported +/-2 sigma for cc=0.
d480_tie_lo = np.where(is_c14, _on_agemodel(p480[0]), d480_tie_age - xerr480 * 2)
d480_tie_hi = np.where(is_c14, _on_agemodel(p480[3]), d480_tie_age + xerr480 * 2)
# matplotlib wants asymmetric xerr as [[distance below], [distance above]]
d480_tie_xerr = np.vstack([d480_tie_age - d480_tie_lo, d480_tie_hi - d480_tie_age])

In [ ]:
# DSDP-480 has two samples at 2390 cm, so `depth` is not a unique index and .interp() fails
# on median480 directly. Drop the duplicate to get an interpolable series — the two ensemble
# columns at 2390 cm carry identical medians (39244.2789 yr), so which one is dropped is moot.
# This is why the round trip through pandas exists; do not "simplify" it away.
new_median = median480.to_series().reset_index().drop_duplicates(subset='depth').set_index('depth').to_xarray()

# DSDP-480/479 splice tie points, interpolated off the canonical age model. age (yr), depth (cm).
d480_479_tiepoint = [[float(new_median.interp(depth=4307).to_array()), 4307], # pollen tie point
                     [float(new_median.interp(depth=4596).to_array()), 4596]] # dDwax tie point
#[110134.71, 4351],

In [ ]:
# DSDP-479 depths
dsdp479_depths=pd.read_excel(f'{dpath1}/sample_depths_479.xlsx').depth.values # cm

# DSDP-479 bacon inputs
dsdp479_input=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP479/DSDP479.csv')
xerr479 = dsdp479_input.error.values  # x-axis error

# DSDP-479 bacon ensemble. its column medians match DSDP479_113_ages.txt to 0.5 yr.
dsdp479_mcmc=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP479/DSDP479_mcmc.csv', header=None)
ens_num=np.linspace(1,len(dsdp479_mcmc),len(dsdp479_mcmc))
dims = ['ensemble_number','depth']
coords = {'ensemble_number': ens_num,
          'depth': dsdp479_depths}
dsdp479_agedepth=xr.DataArray(dsdp479_mcmc.values, dims=dims, coords=coords) # each row is an age model
median479=dsdp479_agedepth.median(axis=0)

# Confidence bands, ordered [2.5, 16, 84, 97.5] — same convention as p480 above.
p479 = np.nanpercentile(dsdp479_mcmc.values, [2.5, 16, 84, 97.5], axis=0)

In [ ]:
# DSDP-479/480 splice tie points, interpolated off the DSDP-479 age model. age (yr), depth (cm).
# 479 depths are unique, so median479 is directly interpolable — no drop_duplicates needed here.
d479_480_tiepoint = [[float(median479.interp(depth=3476)), 3476], # pollen tie point
                     [float(median479.interp(depth=4351)), 4351]] # dDwax tie point

**Mapping other DSDP 480/479 proxy records to our new age-depth model**

Shackleton & Hall, 1982 $\delta^{18}O$<br>
OXYGEN ISOTOPE STUDY OF CONTINUOUS SCRAPE SAMPLES FROM SITE 480<br>
DOI: https://doi.org/10.2973/dsdp.proc.64.165.1982

Keigwin & Jones, 1990 $\delta^{18}O$<br>
Deglacial climatic oscillations in the Gulf of California <br>
DOI: https://doi.org/10.1029/PA005i006p01009

Bryne et al., 1990 pollen<br>
A pollen/dinoflagellate chronology
for DSDP Site 480, Gulf of California <br>
URI: https://hdl.handle.net/1834/31393

Lisiecki & Raymo, 2004 $\delta^{18}O$<br>
A Pliocene-Pleistocene stack of 57 globally distributed benthic δ18O records <br>
DOI: https://doi.org/10.1029/2004PA001071

In [ ]:
# --- Shackleton & Hall d18O --- #
filen=f'{extern}/ShackletonHall82_d18O.xlsx'
sh_d18o={}
depth=pd.read_excel(filen, sheet_name='Sheet1').depth.values
dat=pd.read_excel(filen, sheet_name='Sheet1').d18o.values
sh_d18o['dat'] = xr.DataArray(data=dat,
                              coords={'depth': depth},
                              dims='depth',
                              name='d18o')
sh_d18o['dat'].attrs['source'] = 'Shackleton_Hall_1982'

# --- Keigwin & Jones d18O --- #
filen=f'{extern}/KeigwinJones90_d18O.xlsx'
kj_d18o={}
depth=pd.read_excel(filen, sheet_name='Sheet1').depth.values
dat=pd.read_excel(filen, sheet_name='Sheet1').d18o.values
kj_d18o['dat']=xr.DataArray(data=dat,
                            coords={'depth': depth},
                            dims='depth',
                            name='d18o')
kj_d18o['dat'].attrs['source'] = 'Keigwin_Jones_1990'

# --- Byrne et al. pollen --- #
# the COMBINED Site 480 + Site 479 record.
# Built by scripts/build_combined_pollen_record.py, which splits the Byrne workbook at 4890 cm,
# un-shifts the Site 479 half by the 10 m the workbook applied, dates each core on ITS OWN Bacon
# model, and merges on age. Ages therefore arrive pre-computed, in ka
# --> Do not interpolate to d480 age model here; Re-run build_combined_pollen_record.py if the age models change.
by_aj={}
_pollen = pd.read_csv('data/processed/byrne90_pollen_combined.csv')
_pollen = _pollen[_pollen.dated]              # 97 of 130; the rest sit below their age model
by_aj['age'] = _pollen.age_ka.values          # already ka
by_aj['dat'] = _pollen.artemisia_juniper_pct.values
by_aj['site'] = _pollen.site.values
by_aj['source'] = 'Byrne_1990, Sites 480+479'

# --- LR04 benthic stack --- #
# read from matlab file lr04.mat; `delob` columns are [age (ka), d18O (per mil), error]
delob = sio.loadmat(f'{extern}/lr04.mat')['delob']
lr04={}
lr04['age']=delob[:, 0]
lr04['dat']=delob[:, 1]

hol_d18o_max = 2.1
hol_d18o_min = 2.5

# --- Interp the d18O depths onto the age model --- #
sh_d18o['age']=new_median.interp(depth=sh_d18o['dat'].depth).to_array(name='age').squeeze()
kj_d18o['age']=new_median.interp(depth=kj_d18o['dat'].depth).to_array(name='age').squeeze()


In [ ]:
# --- CREATE DICTIONARY OF AGE MODEL TIE-POINTS --- #
tie_points = {} # One entry per record

# Benthic d18O from THIS study at 48.39 m = 4839 cm. The two values are REPLICATE analyses of
# one sample, so they share a single depth and a single age.
tie_points['d18o_this_study'] = {
    'depth': 4839,                     # cm
    'd18o':  np.array([2.62, 2.43]),   # replicate analyses of one sample
    'err':   0.05,                     # analytical, per mil
}
tie_points['d18o_this_study']['age'] = float(
    _on_agemodel(median480.values, tie_points['d18o_this_study']['depth'])) / 1000

# Shackleton & Hall 1982 benthic d18O, measured at three of THEIR OWN sample depths and placed
# on OUR age model. SH82 dated their 4790 cm sample to 126.082 ka on their 1982 age model;
# our Bacon model puts it at 123.611 ka -- a ~2.5 ka revision, which is what these markers show.
tie_points['sh82'] = {
    'depth': np.array([2395, 3445, 4790]),   # cm, SH82 sample depths
    'd18o':  np.array([3.60, 3.97, 2.48]),
}
tie_points['sh82']['age'] = new_median.interp(
    depth=tie_points['sh82']['depth']).to_array().squeeze().values / 1000   # ka

# LR04 stack values at the TARGET ages of the three Site 480 Bacon tie-points (rows SH-1, SH-2
# and d18O-1 of DSDP480.csv). Separate from sh82 above: these are reference values on the LR04
# curve at the ages Bacon was given, not measurements in our core, and the two are not paired.
tie_points['lr04'] = {
    'age':  np.array([38, 69, 130]),         # ka, as given to Bacon
    'd18o': np.array([4.41, 4.47, 3.67]),
}

# DSDP-479's five Bacon tie-points. Each is a correlation of local dDwax to the LR04 stack, so
# each row carries four paired numbers: the Site 479 depth, the dDwax measured there, the age
# assigned to it, and the LR04 value at that age.
tie_points['dD_479_tie'] = {
    'age':     np.array([    109,     123,     125,     130,     135]),   # ka, as given to Bacon
    'depth_m': np.array([  37.11,   41.81,   43.51,   46.15,   48.11]),
    'dD':      np.array([-157.26, -142.15, -141.77, -149.60, -159.45]),
    'lr04':    np.array([   4.12,    3.10,    3.14,    3.67,    4.86]),
}

# The Site 480 half of the dDwax correlation, paired with the 43.51 m row above.
tie_points['dD_480_tie'] = {'depth_m': 45.96, 'dD': -136.01}

In [ ]:
line_kw={'ls':'-', 'lw':2}
err_line_kw={'ls':':', 'lw':0.75, 'color':'k'}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'clip_on': False, 'zorder':100}
# Filled markers carry a black edge. A marker drawn at the age the MODEL returned (rather than
# the age the model was given) is unfilled: a thin x / + in panels A and B, an open circle in
# panel C. For thin x/+ the stroke colour comes from `color`, so xmark_kw sets no edgecolors.
xmark_kw = {'s': 55, 'linewidths':1.8, 'alpha':1, 'clip_on': False, 'zorder':101}
xtie_kw  = {**scat_kw, 's': 75}   # filled X / P tie-points read too small at s=40
pmark_kw = {**xmark_kw, 's': 70}  # a thin '+' reads smaller than a thin 'x' at equal s
tri_kw   = {**scat_kw, 's': 62}   # ditto the 14C triangles
open_kw  = {'s': 46, 'facecolors':'none', 'edgecolors':'red', 'linewidths':1.6, 'alpha':1, 'clip_on': False, 'zorder':101}
tkw = {'axis':'both', 'direction':'in', 'labelsize': 10}
title_text_kw={'size':15, 'weight':'bold', 'color':'k', 'va':'top', 'ha':'right'}
label_text_kw={'size':11, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
axis_text_kw={'weight':'normal', 'size':11, 'color':'k'}
legend_kw = {'loc':3, 'fontsize':7, 'labelcolor':'k', 'frameon':False}
#settings
xmin=0
xmax=145

fig = plt.figure(figsize=(8,9), constrained_layout=True)

# DSDP-480
ax = plt.subplot2grid((3, 2), (0, 0), rowspan=2, colspan=1)
ax.text(-5000, -300, 'a', **title_text_kw)
# 95% confidence interval
ax.plot(p480[0], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(p480[3], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
# 2-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 p480[0],
                 p480[3],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence')
# 1-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 p480[1],
                 p480[2],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence')               
# median line
ax.plot(median480, dsdp480_agedepth.depth, '-', c='k', lw=1, label='median')
# ms tie-points
plt.errorbar(d480_tie_age[1:6], dsdp480_input.iloc[1:6].depth.values, 
             xerr=d480_tie_xerr[:, 1:6], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[1:6], dsdp480_input.iloc[1:6].depth.values,
            marker='s', color='lightblue', **scat_kw, label='M.S. tie-point')
# planktic 14C, drawn at Bacon's calibrated (calendar) age -- NOT the raw 14C age in the csv.
# See the age-model cell for why, and for what this does and does not demonstrate.
plt.errorbar(d480_tie_age[6:12], dsdp480_input.iloc[6:12].depth.values, 
             xerr=d480_tie_xerr[:, 6:12], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[6:12], dsdp480_input.iloc[6:12].depth.values,
            marker='^', color='white', **tri_kw, label='$^{14}$C$_{planktic}$, calibrated')
# benthic d18O ties. Rows 12-13 are SH-1/SH-2 (correlated to Shackleton & Hall 1982); row 16 is
# this study's own sample at 4839 cm, which had no legend entry at all before. Same evidence type
# and same role in the model, so one symbol and one legend entry. Row 16 is the same sample that
# appears in panel C.
_d18o_ties = [12, 13, 16]
plt.errorbar(d480_tie_age[_d18o_ties], dsdp480_input.iloc[_d18o_ties].depth.values, 
             xerr=d480_tie_xerr[:, _d18o_ties], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[_d18o_ties], dsdp480_input.iloc[_d18o_ties].depth.values,
            marker='o', color='lightblue', **scat_kw, label='$\delta^{18}$O$_{benthic}$ tie-point')
# Cross-core ties at the age they were GIVEN to Bacon. Row 14 (4307 cm) is the pollen
# correlation and row 15 (4596 cm) the dDwax one -- previously both were drawn as one group
# labelled 'dD_C30 tie', which hid the fact that 4307 is a pollen tie.
plt.errorbar(d480_tie_age[14:16], dsdp480_input.iloc[14:16].depth.values, 
             xerr=d480_tie_xerr[:, 14:16], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[15], dsdp480_input.iloc[15].depth,
            marker='X', color='lightblue', **xtie_kw, label='$\delta$D$_{C30}$ tie-point')
plt.scatter(d480_tie_age[14], dsdp480_input.iloc[14].depth,
            marker='P', color='lightblue', **xtie_kw, label='pollen tie-point')
# The same two ties at their MODEL age
plt.scatter(d480_479_tiepoint[1][0], d480_479_tiepoint[1][1],
            marker='x', color='r', **xmark_kw, label='$\delta$D$_{C30}$ correlation 480↔479')
plt.scatter(d480_479_tiepoint[0][0], d480_479_tiepoint[0][1],
            marker='+', color='coral', **pmark_kw, label='pollen correlation 480↔479')
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.text(135000, 100, 'DSDP 480', **title_text_kw)
ax.set(xlim=[1,137000], ylim=[5000,0])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.set_ylabel("Core Depth (cm)")
ax.set_xticks([25000,50000,75000,100000,125000])
ax.set_xticklabels([25,50,75,100,125])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False, **tkw)
ax.legend(**legend_kw)


# DSDP-479
ax = plt.subplot2grid((3, 2), (0, 1), rowspan=2, colspan=1)
ax.text(97500, 3290, 'b', **title_text_kw)
# 2-sigma bounds and shading
ax.plot(p479[0], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(p479[3], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.fill_betweenx(dsdp479_agedepth.depth,
                 p479[0],
                 p479[3],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence interval')
# 1-sigma shading
ax.fill_betweenx(dsdp479_agedepth.depth,
                 p479[1],
                 p479[2],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence interval')
# median line
ax.plot(median479, dsdp479_agedepth.depth, '-', c='k', lw=1, label='median')
# d479-LR04 tie points
# DSDP-479's own tie points, at the ages given to Bacon
plt.errorbar(dsdp479_input.age.values, dsdp479_input.depth.values, 
             xerr=xerr479*2, yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp479_input.age.values, dsdp479_input.depth.values,
            marker='X', color='lightblue', **xtie_kw, label='$\delta$D$_{C30}$ tie-point')
# The DSDP-480 correlations at their MODEL age
plt.scatter(d479_480_tiepoint[1][0], d479_480_tiepoint[1][1],
            marker='x', color='r', **xmark_kw, label='$\delta$D$_{C30}$ correlation 480↔479')
plt.scatter(d479_480_tiepoint[0][0], d479_480_tiepoint[0][1],
            marker='+', color='coral', **pmark_kw, label='pollen correlation 480↔479')
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.text(159000, 3490, 'DSDP 479', **title_text_kw)
ax.set(xlim=[100000,160000], ylim=[5750,3450])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.yaxis.set_label_position('right')
ax.set_ylabel("Core Depth (cm)", rotation=270, labelpad=20)
ax.set_xticks([100000,120000,140000,160000])
ax.set_xticklabels([100,120,140,160])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False,
               left=False, labelleft=False,
               right=True, labelright=True, **tkw)



# --- Interpolated time-series vs. benthic stack --- #

line_kw={'ls':'-', 'lw':1.5}
patch_kw = {'ec':'k', 'lw':1, 'linestyle':':', 'fc':'grey', 'alpha':0.25}
open_kw  = {'s': 46, 'facecolors':'none', 'edgecolors':'red', 'linewidths':1.6, 'alpha':1, 'clip_on': False, 'zorder':101}
tkw = {'axis':'y', 'direction':'out', 'labelsize': 10}
arrow_kw={'arrowstyle':'->', 'color':'grey', 'linewidth':1, 'clip_on':False}
title_text_kw={'size':14, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':7, 'weight':'light', 'color':'grey', 'ha':'left', 'va':'center'}
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'grey'}
sh_line_kw = {**line_kw, 'ls':'-.'}   # SH82 dash-dot
kj_line_kw = {**line_kw, 'ls':':'}    # KJ90 dotted
legend_kw = {'loc':'lower center', 'bbox_to_anchor':(0.65, 0.01), 'ncol':3, 'fontsize':7,
             'labelcolor':'k', 'frameon':False, 'columnspacing':1.1, 'handletextpad':0.6}


ax1 = plt.subplot2grid((3, 2), (2, 0), rowspan=1, colspan=2)
ax1.text(-7.5, 2.1, 'c', **title_text_kw)
# shade expected Holocene d18O values and add leader line annotation
pp=plt.Rectangle((0, hol_d18o_min), 11.7, hol_d18o_max-hol_d18o_min, zorder=100, label='_Hidden', clip_on=True, **patch_kw) 
ax1.add_patch(pp)
ax1.annotate('Expected Holocene\n$\delta^{18}$O$_{benthic}$ for\nGuaymas Basin',
             xy=(11.7,2.2), xytext=(25,2.4),
             arrowprops=arrow_kw, **label_text_kw)
# d18Obenthic curves interpolated to OUR age model
ax1.plot(sh_d18o['age']/1000, sh_d18o['dat'], c='lightblue', **sh_line_kw, label='SH82')
ax1.plot(kj_d18o['age']/1000, kj_d18o['dat'], c='lightsteelblue', **kj_line_kw, label='KJ90')
# benthic stack
ax1.plot(lr04['age'], lr04['dat'], c='k', lw=2, label='LR04') 
# tie-points drawn at the age the MODEL returned for its depth (open red circles)
ax1.scatter([tie_points['d18o_this_study']['age']]*2, tie_points['d18o_this_study']['d18o'],
            marker='o', **open_kw, label='$\\delta^{18}$O$_{benthic}$ tie-point (modeled age)')
ax1.scatter(tie_points['sh82']['age'], tie_points['sh82']['d18o'],
            marker='o', **open_kw, label='_Hidden')
# LR04 value at the age assigned to the three Site 480 benthic d18O ties (38, 69, 130 ka)
# These are reference values on the black LR04 curve, not measurements in our cores
# they mark where each correlation says the core should sit.
ax1.scatter(tie_points['lr04']['age'], tie_points['lr04']['d18o'],
            marker='o', color='red', s=18, edgecolors='none', zorder=100,
            clip_on=False, label='LR04 value at tie-point (input age)')
# settings
ax1.set(xlim=[xmin,xmax], ylim=[5.2,2.1])
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.set_ylabel(u'$\delta^{18}O_{benthic}$ [‰]', labelpad=5, **laxis_text_kw)
ax1.set_xlabel('AGE (ka)')

# Byrne 90 pollen curve interpolated to OUR age model
ax2=ax1.twinx() 
ax2.plot(by_aj['age'], by_aj['dat'], c='grey', ls='-', lw=1.25, label='Byrne90') # art+jun pollen, already in ka
ax2.set(xlim=[xmin,xmax], ylim=[40,-2], yticks=[0,10,20,30])
ax2.minorticks_on()
ax2.tick_params(color='grey', labelcolor='grey', top=False, **tkw)
ax2.set_ylabel(u'%Artemisia+Juniper', labelpad=15, **raxis_text_kw)
ax2.patch.set_visible(False)
ax2.spines['right'].set_color('grey')

# ax2 is a twinx, so its Byrne90 handle has to be merged in by hand.
h1, lb1 = ax1.get_legend_handles_labels()
h2, lb2 = ax2.get_legend_handles_labels()
_h  = [h1[0],  h1[1],  h2[0],  h1[2],  h1[3],  h1[4]]
_lb = [lb1[0], lb1[1], lb2[0], lb1[2], lb1[3], lb1[4]]
ax1.legend(_h, _lb, **legend_kw)

plt.savefig(f'{opath}/for_manuscript/dsdp_480-479.agemodel.pdf', bbox_inches='tight')